In [3]:
import os
import pandas as pd

csv_path = r"../data/processed/Labels_0.6.csv"
image_dir = r"../data/processed/preprocessed_glaucoma"

df = pd.read_csv(csv_path)

missing = []
for img_name in df['Image Name']:
    img_path = os.path.join(image_dir, img_name)
    if not os.path.exists(img_path):
        missing.append(img_name)

print("Missing images:", len(missing))
if len(missing) > 0:
    print(missing[:10])

Missing images: 0


## Split train / val / test

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

csv_path = r"../data/processed/Labels_0.6.csv"
df = pd.read_csv(csv_path)

df = df[['Image Name', 'Label_Binary']].dropna().copy()

train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    stratify=df['Label_Binary'],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=1/3,
    stratify=temp_df['Label_Binary'],
    random_state=42
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

print("\nTrain label distribution:")
print(train_df['Label_Binary'].value_counts())

print("\nValidation label distribution:")
print(val_df['Label_Binary'].value_counts())

print("\nTest label distribution:")
print(test_df['Label_Binary'].value_counts())

Train size: 303
Validation size: 87
Test size: 44

Train label distribution:
Label_Binary
1    214
0     89
Name: count, dtype: int64

Validation label distribution:
Label_Binary
1    61
0    26
Name: count, dtype: int64

Test label distribution:
Label_Binary
1    31
0    13
Name: count, dtype: int64


## Create dataset class

In [5]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset

class GlaucomaDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        image_name = self.dataframe.loc[idx, 'Image Name']
        label = self.dataframe.loc[idx, 'Label_Binary']

        image_path = os.path.join(self.image_dir, image_name)
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(label, dtype=torch.float32)

        return image, label

## Step 4: define transforms

For first training, start simple.

First baseline: no augmentation

In [6]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

## Create DataLoaders

In [7]:
from torch.utils.data import DataLoader

image_dir = r"../data/processed/preprocessed_glaucoma"

train_dataset = GlaucomaDataset(train_df, image_dir, transform=train_transform)
val_dataset = GlaucomaDataset(val_df, image_dir, transform=val_test_transform)
test_dataset = GlaucomaDataset(test_df, image_dir, transform=val_test_transform)

BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 19
Validation batches: 6
Test batches: 3


## Model

In [8]:
import torch
import torch.nn as nn
from torchvision import models

# Hyperparameters
BATCH_SIZE = 16
EPOCHS = 30
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
EARLY_STOP_PATIENCE = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 1)   # binary classification

model = model.to(DEVICE)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Lee Pei En/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:01<00:00, 45.8MB/s]


## Loss and Optimizer

In [ ]:
import torch.optim as optim

criterion = nn.BCEWithLogitsLoss()

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

## Training function

In [10]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    all_labels = []
    all_probs = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        probs = torch.sigmoid(outputs).detach().cpu().numpy()
        all_probs.extend(probs.flatten())

        all_labels.extend(labels.cpu().numpy().flatten())

    epoch_loss = running_loss / len(loader.dataset)

    preds = [1 if p >= 0.5 else 0 for p in all_probs]
    epoch_acc = accuracy_score(all_labels, preds)

    return epoch_loss, epoch_acc

## Validation function

In [11]:
def evaluate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            probs = torch.sigmoid(outputs).cpu().numpy()
            all_probs.extend(probs.flatten())

            all_labels.extend(labels.cpu().numpy().flatten())

    epoch_loss = running_loss / len(loader.dataset)

    preds = [1 if p >= 0.5 else 0 for p in all_probs]
    epoch_acc = accuracy_score(all_labels, preds)

    return epoch_loss, epoch_acc



## Training model

In [12]:
import copy
from sklearn.metrics import accuracy_score

best_val_loss = float("inf")
best_model_wts = copy.deepcopy(model.state_dict())
patience_counter = 0

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

print("\nStarting training...\n")

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc = evaluate_one_epoch(model, val_loader, criterion, DEVICE)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f"Epoch [{epoch+1}/{EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print("-" * 50)

    # Early stopping check
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        patience_counter = 0
        print("Validation loss improved. Model saved.\n")
    else:
        patience_counter += 1
        print(f"No improvement. Early stop counter: {patience_counter}/{EARLY_STOP_PATIENCE}\n")

        if patience_counter >= EARLY_STOP_PATIENCE:
            print("Early stopping triggered.")
            break


Starting training...

Epoch [1/30]
Train Loss: 0.2322 | Train Acc: 0.9076
Val   Loss: 0.1252 | Val   Acc: 0.9425
--------------------------------------------------
Validation loss improved. Model saved.

Epoch [2/30]
Train Loss: 0.0745 | Train Acc: 0.9736
Val   Loss: 0.1612 | Val   Acc: 0.9425
--------------------------------------------------
No improvement. Early stop counter: 1/5

Epoch [3/30]
Train Loss: 0.0397 | Train Acc: 0.9934
Val   Loss: 0.1572 | Val   Acc: 0.9310
--------------------------------------------------
No improvement. Early stop counter: 2/5

Epoch [4/30]
Train Loss: 0.0544 | Train Acc: 0.9868
Val   Loss: 0.1809 | Val   Acc: 0.9540
--------------------------------------------------
No improvement. Early stop counter: 3/5

Epoch [5/30]
Train Loss: 0.0227 | Train Acc: 0.9934
Val   Loss: 0.2301 | Val   Acc: 0.9310
--------------------------------------------------
No improvement. Early stop counter: 4/5

Epoch [6/30]
Train Loss: 0.0269 | Train Acc: 0.9868
Val   Loss:

## Load best model

In [13]:
model.load_state_dict(best_model_wts)
print("Best model loaded.")

Best model loaded.


## Evaluate test set

In [14]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

model.eval()

all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE).unsqueeze(1)

        outputs = model(images)
        probs = torch.sigmoid(outputs).cpu().numpy()

        all_probs.extend(probs.flatten())
        all_labels.extend(labels.cpu().numpy().flatten())

all_preds = [1 if p >= 0.5 else 0 for p in all_probs]

test_accuracy = accuracy_score(all_labels, all_preds)
test_precision = precision_score(all_labels, all_preds, zero_division=0)
test_recall = recall_score(all_labels, all_preds, zero_division=0)
test_f1 = f1_score(all_labels, all_preds, zero_division=0)

# ROC-AUC needs both classes present
if len(set(all_labels)) > 1:
    test_auc = roc_auc_score(all_labels, all_probs)
else:
    test_auc = None

cm = confusion_matrix(all_labels, all_preds)

print("\n================ TEST RESULTS ================")
print(f"Test Accuracy : {test_accuracy:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall   : {test_recall:.4f}")
print(f"Test F1-score : {test_f1:.4f}")

if test_auc is not None:
    print(f"Test ROC-AUC  : {test_auc:.4f}")
else:
    print("Test ROC-AUC  : Cannot compute (only one class present)")

print("\nConfusion Matrix:")
print(cm)


================ TEST RESULTS ================
Test Accuracy : 0.9545
Test Precision: 1.0000
Test Recall   : 0.9355
Test F1-score : 0.9667
Test ROC-AUC  : 1.0000

Confusion Matrix:
[[13  0]
 [ 2 29]]


## Save best model

In [15]:
model_save_path = r"../models/glaucoma_resnet18_best.pth"
torch.save(model.state_dict(), model_save_path)
print(f"\nBest model saved to: {model_save_path}")


Best model saved to: ../models/glaucoma_resnet18_best.pth
